In [ ]:
# ============================================================
# RAIS_BASE_MODELO_FINAL_V3 — v3
# CONSTRUÇÃO, ENRIQUECIMENTO E VALIDAÇÃO
# ============================================================
#
# REGRAS CONFIRMADAS NA AUDITORIA GLOBAL V2 x FONTE
# ------------------------------------------------------------
# - 36/36 arquivos com exatamente o mesmo número de linhas.
# - UF: 19.553.768/19.553.768 iguais linha a linha.
# - Família CBO: 19.553.768/19.553.768 iguais linha a linha.
# - Tempo de emprego: 19.553.768/19.553.768 iguais numericamente.
# - Sexo: mesma categoria; em 2020–2022 há apenas zero à esquerda.
# - Idade: valores menores que 16 foram convertidos para NaN na V2.
# - Horas: 0 e valores acima de 44 foram convertidos para NaN na V2.
#
# ESTRATÉGIA
# ------------------------------------------------------------
# A V3 NÃO reconstrói o Y_doenca.
#
# Ela:
#   1. recupera os professores das 5 famílias CBO em 2020–2025;
#   2. preserva uma fonte enriquecida com as colunas originais da RAIS;
#   3. usa a BASE FINAL V2 como "espinha dorsal";
#   4. valida o alinhamento linha a linha entre a fonte e a V2;
#      em horas contratuais, 0 na fonte é aceito como equivalente a NaN
#      na V2, e essa equivalência é contabilizada na auditoria;
#   5. somente após alinhamento perfeito acrescenta novas covariáveis;
#   6. copia Y_doenca e todas as variáveis já validadas DIRETAMENTE da V2;
#   7. grava RAIS_BASE_MODELO_FINAL_V3;
#   8. compara V2 e V3 por arquivo, ano, UF, família CBO e Y_doenca.
#
# AFASTAMENTOS
# ------------------------------------------------------------
# Causa do afastamento e quantidade de dias NÃO são acrescentadas à V3.
# O alvo Y_doenca é copiado da V2 para impedir alteração do desfecho e
# evitar que campos diretamente ligados ao afastamento sejam usados
# acidentalmente como preditores.
#
# NOVAS VARIÁVEIS HARMONIZADAS
# ------------------------------------------------------------
# - Municipio_estabelecimento_codigo
# - Municipio_trabalhador_codigo        [preservada, não prevista como principal]
# - CBO_6digitos
# - CNAE_2_classe_codigo
# - CNAE_2_subclasse_codigo
# - IBGE_subsetor_codigo
# - Tipo_admissao_codigo
# - Mes_admissao_codigo
# - Tipo_estabelecimento_codigo
# - Indicador_trabalho_parcial_codigo
# - Indicador_trabalho_intermitente_codigo
# - Vinculo_ativo_31_12_codigo
# - Mes_desligamento_codigo             [preservar, não usar inicialmente]
# - Motivo_desligamento_codigo          [preservar, não usar inicialmente]
# - Remuneracao_media_nominal           [preservar, não usar inicialmente]
# - Remuneracao_dezembro_nominal        [preservar, não usar inicialmente]
# - Indicador_vinculo_abandonado_codigo [2023–2025; preservar]
# - Categoria_trabalhador_codigo        [2023–2025; preservar]
#
# IMPORTANTE
# ------------------------------------------------------------
# - Os Parquets de origem 2020–2025 são filtrados pelas CINCO famílias:
#       2312, 2313, 2321, 3312 e 3321.
# - 2022 já havia sido validado com 2.966.255 registros, exatamente igual à V2.
# - O código é retomável: arquivos válidos já concluídos podem ser reutilizados.
# - Nenhum arquivo da V2 é alterado.
# ============================================================


# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

from google.colab import drive
from ftplib import FTP
from urllib.parse import quote

from collections import Counter

import gc
import glob
import hashlib
import json
import os
import re
import shutil
import subprocess
import unicodedata

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from IPython.display import display


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 3. CONFIGURAÇÕES
# ============================================================

PASTA_TCC = (
    "/content/drive/MyDrive/TCC_2"
)

PASTA_DADOS = os.path.join(
    PASTA_TCC,
    "dados"
)

# Base final já validada.
PASTA_V2 = os.path.join(
    PASTA_DADOS,
    "RAIS_BASE_MODELO_FINAL_V2"
)

# Fonte com 5 famílias. Para os anos ainda não existentes, será criada.
PASTA_FONTE_V3 = os.path.join(
    PASTA_DADOS,
    "RAIS_PROFESSORES_V3_FONTE"
)

# 2022 já criado e validado na etapa anterior.
PASTA_CANDIDATA_2022 = os.path.join(
    PASTA_DADOS,
    "RAIS_PROFESSORES_V3_CANDIDATA",
    "2022"
)

# Saída definitiva da nova base de modelagem.
PASTA_V3 = os.path.join(
    PASTA_DADOS,
    "RAIS_BASE_MODELO_FINAL_V3"
)

PASTA_RESULTADOS = os.path.join(
    PASTA_TCC,
    "resultados",
    "CONSTRUCAO_RAIS_BASE_MODELO_FINAL_V3"
)

# Onde procurar / guardar compactados.
PASTA_COMPACTADOS = os.path.join(
    PASTA_DADOS,
    "RAIS_OUTROS_ESTADOS"
)

# Temporários locais do Colab.
PASTA_TEMP = (
    "/content/RAIS_V3_TEMP"
)

for pasta in [
    PASTA_FONTE_V3,
    PASTA_V3,
    PASTA_RESULTADOS,
    PASTA_COMPACTADOS,
    PASTA_TEMP,
]:
    os.makedirs(
        pasta,
        exist_ok=True
    )


ANOS = [
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
]


GRUPOS = {
    "NORTE":
        "RAIS_VINC_PUB_NORTE",

    "NORDESTE":
        "RAIS_VINC_PUB_NORDESTE",

    "CENTRO_OESTE":
        "RAIS_VINC_PUB_CENTRO_OESTE",

    "MG_ES_RJ":
        "RAIS_VINC_PUB_MG_ES_RJ",

    "SP":
        "RAIS_VINC_PUB_SP",

    "SUL":
        "RAIS_VINC_PUB_SUL",
}


FAMILIAS_CBO = {

    "2312":
        "Professores de nível superior do ensino fundamental - anos iniciais",

    "2313":
        "Professores de nível superior do ensino fundamental - anos finais",

    "2321":
        "Professores do ensino médio",

    "3312":
        "Professores de nível médio no ensino fundamental",

    "3321":
        "Professores leigos no ensino fundamental",
}


MAPA_UF = {

    "11": "RO",
    "12": "AC",
    "13": "AM",
    "14": "RR",
    "15": "PA",
    "16": "AP",
    "17": "TO",

    "21": "MA",
    "22": "PI",
    "23": "CE",
    "24": "RN",
    "25": "PB",
    "26": "PE",
    "27": "AL",
    "28": "SE",
    "29": "BA",

    "31": "MG",
    "32": "ES",
    "33": "RJ",
    "35": "SP",

    "41": "PR",
    "42": "SC",
    "43": "RS",

    "50": "MS",
    "51": "MT",
    "52": "GO",
    "53": "DF",
}


UFS_ESPERADAS = {

    "NORTE": {
        "AC", "AP", "AM", "PA",
        "RO", "RR", "TO"
    },

    "NORDESTE": {
        "AL", "BA", "CE", "MA",
        "PB", "PE", "PI", "RN", "SE"
    },

    "CENTRO_OESTE": {
        "DF", "GO", "MT", "MS"
    },

    "MG_ES_RJ": {
        "MG", "ES", "RJ"
    },

    "SP": {
        "SP"
    },

    "SUL": {
        "PR", "SC", "RS"
    },
}


HOST = "ftp.mtps.gov.br"
BASE_FTP = "/pdet/microdados/RAIS"

CHUNKSIZE_RAW = 300_000
BATCH_SIZE_PARQUET = 200_000

# False = reutiliza arquivos já concluídos.
SOBRESCREVER_FONTE = False
SOBRESCREVER_V3 = True

# Reutiliza o 2022 já validado, evitando processamento redundante.
REUTILIZAR_2022_VALIDADO = True


# ============================================================
# 4. COLUNAS NOVAS DA V3
# ============================================================

NOVAS_COLUNAS = [
    "Municipio_estabelecimento_codigo",
    "Municipio_trabalhador_codigo",
    "CBO_6digitos",
    "CNAE_2_classe_codigo",
    "CNAE_2_subclasse_codigo",
    "IBGE_subsetor_codigo",
    "Tipo_admissao_codigo",
    "Mes_admissao_codigo",
    "Tipo_estabelecimento_codigo",
    "Indicador_trabalho_parcial_codigo",
    "Indicador_trabalho_intermitente_codigo",
    "Vinculo_ativo_31_12_codigo",
    "Mes_desligamento_codigo",
    "Motivo_desligamento_codigo",
    "Remuneracao_media_nominal",
    "Remuneracao_dezembro_nominal",
    "Indicador_vinculo_abandonado_codigo",
    "Categoria_trabalhador_codigo",
]


# ============================================================
# 5. INSTALAR 7ZIP SE NECESSÁRIO
# ============================================================

if shutil.which(
    "7z"
) is None:

    subprocess.run(
        [
            "apt-get",
            "update"
        ],
        stdout=subprocess.DEVNULL,
        check=True
    )

    subprocess.run(
        [
            "apt-get",
            "install",
            "-y",
            "p7zip-full"
        ],
        stdout=subprocess.DEVNULL,
        check=True
    )


# ============================================================
# 6. FUNÇÕES DE NORMALIZAÇÃO
# ============================================================

def normalizar_texto(
    texto
):

    texto = unicodedata.normalize(
        "NFKD",
        str(
            texto
        )
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(
            caractere
        )
    )

    texto = texto.upper()

    texto = re.sub(
        r"[^A-Z0-9]+",
        " ",
        texto
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    ).strip()

    return texto


def normalizar_codigo(
    serie
):

    return (
        serie
        .astype(
            "string"
        )
        .str.strip()
        .replace(
            {
                "":
                    pd.NA,

                "NAN":
                    pd.NA,

                "None":
                    pd.NA,
            }
        )
    )


def extrair_digitos(
    serie,
    minimo=1,
    maximo=None
):

    s = (
        serie
        .astype(
            "string"
        )
        .str.strip()
    )

    if maximo is None:

        padrao = (
            rf"(\d{{{minimo},}})"
        )

    elif minimo == maximo:

        padrao = (
            rf"(\d{{{minimo}}})"
        )

    else:

        padrao = (
            rf"(\d{{{minimo},{maximo}}})"
        )

    return (
        s
        .str.extract(
            padrao,
            expand=False
        )
    )


def para_numero(
    serie
):
    """
    Converte números sem corromper valores já em formato decimal
    com ponto.

    Exemplos:
        "62"       -> 62
        "62.0"     -> 62.0
        "1234,56"  -> 1234.56
        "1.234,56" -> 1234.56
    """

    s = (
        serie
        .astype("string")
        .str.strip()
        .replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "<NA>": pd.NA,
            }
        )
    )

    # 1) Tenta primeiro o formato numérico padrão/Python,
    #    preservando corretamente valores como "62.0".
    direto = pd.to_numeric(
        s,
        errors="coerce"
    )

    # 2) Somente os valores que falharam são tratados
    #    como representação brasileira:
    #       1.234,56 -> 1234.56
    precisa_br = (
        direto.isna()
        &
        s.notna()
    )

    if precisa_br.any():

        s_br = (
            s.loc[
                precisa_br
            ]
            .str.replace(
                ".",
                "",
                regex=False
            )
            .str.replace(
                ",",
                ".",
                regex=False
            )
        )

        direto.loc[
            precisa_br
        ] = pd.to_numeric(
            s_br,
            errors="coerce"
        )

    return direto


def normalizar_string_comparacao(
    serie
):

    return (
        serie
        .astype(
            "string"
        )
        .str.strip()
        .str.upper()
        .fillna(
            "__NA__"
        )
    )


def detectar_coluna(
    colunas,
    candidatos_exatos=None,
    contem_todos=None,
    obrigatoria=False,
    descricao=""
):

    candidatos_exatos = (
        candidatos_exatos
        or []
    )

    contem_todos = (
        contem_todos
        or []
    )

    mapa = {
        normalizar_texto(
            coluna
        ):
            coluna

        for coluna
        in colunas
    }


    for candidato in candidatos_exatos:

        chave = normalizar_texto(
            candidato
        )

        if chave in mapa:

            return mapa[
                chave
            ]


    for coluna in colunas:

        nome = normalizar_texto(
            coluna
        )

        for termos in contem_todos:

            if all(
                normalizar_texto(
                    termo
                )
                in
                nome

                for termo
                in termos
            ):

                return coluna


    if obrigatoria:

        raise RuntimeError(
            "Não encontrei a coluna obrigatória "
            f"'{descricao}'.\n"
            f"Colunas disponíveis:\n{colunas}"
        )

    return None


# ============================================================
# 7. DETECTAR LAYOUT DA RAIS 2020–2025
# ============================================================

def detectar_layout_colunas(
    colunas
):

    layout = {}


    layout[
        "cbo"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "CBO Ocupação 2002",
            "CBO 2002 Ocupação - Código",
        ],
        contem_todos=[
            [
                "CBO",
                "2002",
                "OCUP"
            ]
        ],
        obrigatoria=True,
        descricao="CBO 2002"
    )


    layout[
        "municipio_estab"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Município",
            "Município - Código",
            "Municipio",
            "Municipio - Codigo",
        ],
        obrigatoria=True,
        descricao="Município do estabelecimento"
    )


    layout[
        "municipio_trab"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Mun Trab",
            "Município Trab - Código",
            "Município Trab",
            "Municipio Trab",
        ],
        contem_todos=[
            [
                "MUN",
                "TRAB"
            ]
        ],
        obrigatoria=False,
        descricao="Município do trabalhador"
    )


    layout[
        "idade"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Idade"
        ],
        obrigatoria=True,
        descricao="Idade"
    )


    layout[
        "horas"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Qtd Hora Contr"
        ],
        contem_todos=[
            [
                "QTD",
                "HORA",
                "CONTR"
            ]
        ],
        obrigatoria=True,
        descricao="Quantidade de horas contratuais"
    )


    layout[
        "tempo_emprego"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Tempo Emprego"
        ],
        contem_todos=[
            [
                "TEMPO",
                "EMPREGO"
            ]
        ],
        obrigatoria=True,
        descricao="Tempo de emprego"
    )


    layout[
        "sexo"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Sexo Trabalhador",
            "Sexo - Código",
            "Sexo"
        ],
        contem_todos=[
            [
                "SEXO"
            ]
        ],
        obrigatoria=True,
        descricao="Sexo"
    )


    layout[
        "cnae_classe"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "CNAE 2.0 Classe",
            "CNAE 2.0 Classe - Código",
        ],
        contem_todos=[
            [
                "CNAE",
                "2",
                "CLASSE"
            ]
        ],
        obrigatoria=True,
        descricao="CNAE 2.0 Classe"
    )


    layout[
        "cnae_subclasse"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "CNAE 2.0 Subclasse",
            "CNAE 2.0 Subclasse - Codigo",
            "CNAE 2.0 Subclasse - Código",
        ],
        contem_todos=[
            [
                "CNAE",
                "2",
                "SUBCLASSE"
            ]
        ],
        obrigatoria=True,
        descricao="CNAE 2.0 Subclasse"
    )


    layout[
        "ibge_subsetor"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "IBGE Subsetor",
            "IBGE Subsetor - Código",
        ],
        contem_todos=[
            [
                "IBGE",
                "SUBSETOR"
            ]
        ],
        obrigatoria=True,
        descricao="IBGE Subsetor"
    )


    layout[
        "tipo_admissao"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Tipo Admissão",
            "Tipo Admissão Trabalhador - Código",
        ],
        contem_todos=[
            [
                "TIPO",
                "ADMIS"
            ]
        ],
        obrigatoria=True,
        descricao="Tipo de admissão"
    )


    layout[
        "mes_admissao"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Mês Admissão",
            "Mês Admissão - Código",
        ],
        contem_todos=[
            [
                "MES",
                "ADMIS"
            ]
        ],
        obrigatoria=True,
        descricao="Mês de admissão"
    )


    layout[
        "tipo_estabelecimento"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Tipo Estab",
            "Tipo Estabelecimento - Código",
        ],
        contem_todos=[
            [
                "TIPO",
                "ESTAB"
            ]
        ],
        obrigatoria=True,
        descricao="Tipo do estabelecimento"
    )


    layout[
        "trab_parcial"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Ind Trab Parcial",
            "Ind Trabalho Parcial - Código",
        ],
        contem_todos=[
            [
                "TRAB",
                "PARCIAL"
            ]
        ],
        obrigatoria=False,
        descricao="Indicador de trabalho parcial"
    )


    layout[
        "trab_intermitente"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Ind Trab Intermitente",
            "Ind Trabalho Intermitente - Código",
        ],
        contem_todos=[
            [
                "TRAB",
                "INTERMIT"
            ]
        ],
        obrigatoria=False,
        descricao="Indicador de trabalho intermitente"
    )


    layout[
        "vinculo_ativo"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Vínculo Ativo 31/12",
            "Ind Vínculo Ativo 31/12 - Código",
        ],
        contem_todos=[
            [
                "VINCULO",
                "ATIVO",
                "31"
            ]
        ],
        obrigatoria=False,
        descricao="Vínculo ativo em 31/12"
    )


    layout[
        "mes_desligamento"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Mês Desligamento",
            "Mês Desligamento - Código",
        ],
        contem_todos=[
            [
                "MES",
                "DESLIG"
            ]
        ],
        obrigatoria=False,
        descricao="Mês de desligamento"
    )


    layout[
        "motivo_desligamento"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Motivo Desligamento",
            "Motivo Desligamento - Código",
        ],
        contem_todos=[
            [
                "MOTIVO",
                "DESLIG"
            ]
        ],
        obrigatoria=False,
        descricao="Motivo de desligamento"
    )


    layout[
        "rem_media_nominal"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Vl Remun Média Nom",
            "Vl Rem Média Nom",
        ],
        contem_todos=[
            [
                "VL",
                "REM",
                "MEDIA",
                "NOM"
            ]
        ],
        obrigatoria=False,
        descricao="Remuneração média nominal"
    )


    layout[
        "rem_dezembro_nominal"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Vl Remun Dezembro Nom",
            "Vl Rem Dezembro Nom",
        ],
        contem_todos=[
            [
                "VL",
                "REM",
                "DEZ",
                "NOM"
            ]
        ],
        obrigatoria=False,
        descricao="Remuneração de dezembro nominal"
    )


    layout[
        "vinculo_abandonado"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Ind Vínculo Abandonado - Código",
        ],
        contem_todos=[
            [
                "VINCULO",
                "ABANDON"
            ]
        ],
        obrigatoria=False,
        descricao="Indicador de vínculo abandonado"
    )


    layout[
        "categoria_trabalhador"
    ] = detectar_coluna(
        colunas,
        candidatos_exatos=[
            "Categoria Trabalhador - Código",
        ],
        contem_todos=[
            [
                "CATEGORIA",
                "TRABALHADOR"
            ]
        ],
        obrigatoria=False,
        descricao="Categoria do trabalhador"
    )


    return layout


# ============================================================
# 8. DETECTAR FORMATO DO ARQUIVO BRUTO
# ============================================================

def identificar_formato_bruto(
    caminho
):

    for encoding in [
        "cp1252",
        "latin1",
        "utf-8",
    ]:

        for sep in [
            ";",
            ",",
            "\t",
            "|",
        ]:

            try:

                cab = pd.read_csv(
                    caminho,
                    sep=sep,
                    encoding=encoding,
                    nrows=0
                )

                if len(
                    cab.columns
                ) < 20:

                    continue

                layout = detectar_layout_colunas(
                    list(
                        cab.columns
                    )
                )

                return {
                    "encoding":
                        encoding,

                    "sep":
                        sep,

                    "colunas":
                        list(
                            cab.columns
                        ),

                    "layout":
                        layout,
                }

            except RuntimeError:
                raise

            except Exception:
                continue


    raise RuntimeError(
        "Não foi possível detectar o formato de "
        f"{caminho}"
    )


# ============================================================
# 9. LOCALIZAR ARQUIVO BRUTO LOCAL
# ============================================================

def localizar_bruto_local(
    ano,
    nome_base
):

    candidatos = glob.glob(
        os.path.join(
            PASTA_DADOS,
            "**",
            f"{nome_base}*"
        ),
        recursive=True
    )


    validos = []


    for caminho in candidatos:

        if not os.path.isfile(
            caminho
        ):

            continue


        nome = (
            os.path.basename(
                caminho
            )
            .lower()
        )


        if nome.endswith(
            (
                ".7z",
                ".zip",
                ".parquet",
                ".csv",
            )
        ):

            # .csv aqui pode ser relatório, não a RAIS bruta.
            continue


        partes = (
            os.path.normpath(
                caminho
            )
            .split(
                os.sep
            )
        )


        if str(
            ano
        ) not in partes:

            continue


        validos.append(
            caminho
        )


    if not validos:

        return None


    return max(
        validos,
        key=os.path.getsize
    )


# ============================================================
# 10. FTP / COMPACTADOS COMO FALLBACK
# ============================================================

_cache_ftp = {}


def listar_ftp(
    ano
):

    if ano in _cache_ftp:

        return _cache_ftp[
            ano
        ]


    erros = []


    for encoding in [
        "utf-8",
        "cp1252",
        "latin1",
    ]:

        ftp = None

        try:

            ftp = FTP(
                HOST,
                timeout=180,
                encoding=encoding
            )

            ftp.login()

            ftp.cwd(
                f"{BASE_FTP}/{ano}"
            )

            arquivos = ftp.nlst()

            ftp.quit()

            _cache_ftp[
                ano
            ] = arquivos

            return arquivos

        except Exception as erro:

            erros.append(
                f"{encoding}: {erro}"
            )

            try:

                if ftp is not None:

                    ftp.close()

            except Exception:
                pass


    raise RuntimeError(
        "Não foi possível listar o FTP:\n"
        +
        "\n".join(
            erros
        )
    )


def localizar_compactado_local(
    ano,
    nome_base
):

    candidatos = glob.glob(
        os.path.join(
            PASTA_DADOS,
            "**",
            f"*{nome_base}*.7z"
        ),
        recursive=True
    )


    candidatos = [
        caminho

        for caminho
        in candidatos

        if str(
            ano
        )
        in
        os.path.normpath(
            caminho
        )
        .split(
            os.sep
        )
    ]


    if not candidatos:

        return None


    return max(
        candidatos,
        key=os.path.getsize
    )


def obter_compactado(
    ano,
    grupo,
    nome_base
):

    local = localizar_compactado_local(
        ano,
        nome_base
    )


    if local is not None:

        return local


    arquivos = listar_ftp(
        ano
    )


    candidatos = [

        arquivo

        for arquivo
        in arquivos

        if (
            nome_base.upper()
            in
            os.path.basename(
                arquivo
            )
            .upper()
        )

        and

        os.path.basename(
            arquivo
        )
        .upper()
        .endswith(
            ".7Z"
        )
    ]


    if len(
        candidatos
    ) != 1:

        raise RuntimeError(
            f"{ano} | {grupo}: esperado 1 .7z, "
            f"encontrados {candidatos}"
        )


    remoto = candidatos[
        0
    ]


    nome_remoto = os.path.basename(
        remoto
    )


    pasta_destino = os.path.join(
        PASTA_COMPACTADOS,
        grupo,
        str(
            ano
        )
    )


    os.makedirs(
        pasta_destino,
        exist_ok=True
    )


    destino = os.path.join(
        pasta_destino,
        nome_remoto
    )


    url = (
        f"ftp://{HOST}"
        f"{BASE_FTP}/{ano}/"
        f"{quote(nome_remoto)}"
    )


    print(
        "Baixando:"
    )

    print(
        url
    )


    subprocess.run(
        [
            "wget",
            "--continue",
            "--progress=bar:force",
            "-O",
            destino,
            url,
        ],
        check=True
    )


    return destino


def extrair_compactado(
    arquivo_7z,
    ano,
    grupo
):

    destino = os.path.join(
        PASTA_TEMP,
        str(
            ano
        ),
        grupo
    )


    if os.path.exists(
        destino
    ):

        shutil.rmtree(
            destino
        )


    os.makedirs(
        destino,
        exist_ok=True
    )


    subprocess.run(
        [
            "7z",
            "x",
            arquivo_7z,
            f"-o{destino}",
            "-y",
        ],
        stdout=subprocess.DEVNULL,
        check=True
    )


    candidatos = []


    for raiz, _, nomes in os.walk(
        destino
    ):

        for nome in nomes:

            caminho = os.path.join(
                raiz,
                nome
            )

            if os.path.isfile(
                caminho
            ):

                candidatos.append(
                    caminho
                )


    if len(
        candidatos
    ) != 1:

        raise RuntimeError(
            f"{ano} | {grupo}: extração gerou "
            f"{len(candidatos)} arquivos: "
            f"{candidatos}"
        )


    return candidatos[
        0
    ]


# ============================================================
# 11. CRIAR FONTE ENRIQUECIDA DE 5 FAMÍLIAS
# ============================================================

def fonte_v3_valida(
    caminho
):

    if not os.path.exists(
        caminho
    ):

        return False


    try:

        pf = pq.ParquetFile(
            caminho
        )

        obrigatorias = {
            "CBO_padronizada",
            "Familia_CBO",
            "UF",
            "Municipio_estabelecimento_codigo",
            "Ano",
            "Grupo_Origem",
        }

        return (
            pf.metadata.num_rows
            >
            0

            and

            obrigatorias.issubset(
                set(
                    pf.schema_arrow.names
                )
            )
        )

    except Exception:

        return False


def processar_fonte_cinco_familias(
    ano,
    grupo,
    bruto,
    saida
):

    formato = identificar_formato_bruto(
        bruto
    )

    layout = formato[
        "layout"
    ]


    print(
        f"Formato: {formato['encoding']} | "
        f"sep={repr(formato['sep'])} | "
        f"{len(formato['colunas'])} colunas"
    )


    tmp = (
        saida
        +
        ".tmp"
    )


    if os.path.exists(
        tmp
    ):

        os.remove(
            tmp
        )


    writer = None

    linhas_brutas = 0

    linhas_filtradas = 0

    cont_familias = Counter()

    cont_ufs = Counter()


    try:

        leitor = pd.read_csv(
            bruto,
            sep=formato[
                "sep"
            ],
            encoding=formato[
                "encoding"
            ],
            dtype=str,
            chunksize=CHUNKSIZE_RAW,
            low_memory=False,
            on_bad_lines="error",
        )


        for numero_chunk, chunk in enumerate(
            leitor,
            start=1
        ):

            linhas_brutas += len(
                chunk
            )


            cbo = extrair_digitos(
                chunk[
                    layout[
                        "cbo"
                    ]
                ],
                minimo=6,
                maximo=6
            )


            familia = cbo.str[
                :4
            ]


            mascara = familia.isin(
                FAMILIAS_CBO.keys()
            )


            prof = (
                chunk
                .loc[
                    mascara
                ]
                .copy()
            )


            if prof.empty:

                del (
                    chunk,
                    cbo,
                    familia,
                    mascara
                )

                gc.collect()

                continue


            prof[
                "CBO_padronizada"
            ] = (
                cbo
                .loc[
                    mascara
                ]
                .values
            )


            prof[
                "Familia_CBO"
            ] = (
                familia
                .loc[
                    mascara
                ]
                .values
            )


            prof[
                "Descricao_Familia_CBO"
            ] = (
                prof[
                    "Familia_CBO"
                ]
                .map(
                    FAMILIAS_CBO
                )
            )


            prof[
                "Ano"
            ] = str(
                ano
            )


            prof[
                "Grupo_Origem"
            ] = grupo


            mun_estab = extrair_digitos(
                prof[
                    layout[
                        "municipio_estab"
                    ]
                ],
                minimo=6,
                maximo=6
            )


            prof[
                "Municipio_estabelecimento_codigo"
            ] = mun_estab


            prof[
                "UF_estabelecimento"
            ] = (
                mun_estab
                .str[:2]
                .map(
                    MAPA_UF
                )
            )


            prof[
                "UF"
            ] = (
                prof[
                    "UF_estabelecimento"
                ]
            )


            if (
                layout[
                    "municipio_trab"
                ]
                is not None
            ):

                mun_trab = extrair_digitos(
                    prof[
                        layout[
                            "municipio_trab"
                        ]
                    ],
                    minimo=6,
                    maximo=6
                )


                prof[
                    "Municipio_trabalhador_codigo"
                ] = mun_trab


                prof[
                    "UF_trabalhador"
                ] = (
                    mun_trab
                    .str[:2]
                    .map(
                        MAPA_UF
                    )
                )

            else:

                prof[
                    "Municipio_trabalhador_codigo"
                ] = pd.NA

                prof[
                    "UF_trabalhador"
                ] = pd.NA


            # Mantemos todas as colunas como string nessa fonte.
            # A V3 de modelagem terá tipos mais adequados.
            for coluna in prof.columns:

                prof[
                    coluna
                ] = (
                    prof[
                        coluna
                    ]
                    .astype(
                        "string"
                    )
                )


            tabela = pa.Table.from_pandas(
                prof,
                preserve_index=False
            )


            if writer is None:

                writer = pq.ParquetWriter(
                    tmp,
                    tabela.schema,
                    compression="snappy"
                )


            writer.write_table(
                tabela
            )


            linhas_filtradas += len(
                prof
            )


            for fam, qtd in (
                prof[
                    "Familia_CBO"
                ]
                .value_counts()
                .items()
            ):

                cont_familias[
                    str(
                        fam
                    )
                ] += int(
                    qtd
                )


            for uf, qtd in (
                prof[
                    "UF"
                ]
                .value_counts()
                .items()
            ):

                cont_ufs[
                    str(
                        uf
                    )
                ] += int(
                    qtd
                )


            print(
                f"Chunk {numero_chunk}: "
                f"{len(chunk):,} brutos | "
                f"{len(prof):,} professores | "
                f"acumulado={linhas_filtradas:,}"
            )


            del (
                chunk,
                prof,
                tabela,
                cbo,
                familia,
                mascara,
                mun_estab
            )


            if (
                "mun_trab"
                in locals()
            ):

                del mun_trab


            gc.collect()


    finally:

        if writer is not None:

            writer.close()


    if not os.path.exists(
        tmp
    ):

        raise RuntimeError(
            f"{ano} | {grupo}: nenhum registro filtrado."
        )


    pf = pq.ParquetFile(
        tmp
    )


    if int(
        pf.metadata.num_rows
    ) != linhas_filtradas:

        raise RuntimeError(
            f"{ano} | {grupo}: divergência de gravação "
            f"{linhas_filtradas:,} vs "
            f"{pf.metadata.num_rows:,}."
        )


    if os.path.exists(
        saida
    ):

        os.remove(
            saida
        )


    os.replace(
        tmp,
        saida
    )


    return {
        "Ano":
            ano,

        "Grupo":
            grupo,

        "Linhas_brutas":
            linhas_brutas,

        "Professores_5_familias":
            linhas_filtradas,

        "Qtd_colunas_fonte":
            len(
                pq.ParquetFile(
                    saida
                ).schema_arrow.names
            ),

        "Familias_encontradas":
            ", ".join(
                sorted(
                    cont_familias.keys()
                )
            ),

        "UFs_encontradas":
            ", ".join(
                sorted(
                    cont_ufs.keys()
                )
            ),

        "Arquivo_fonte":
            saida,
    }


# ============================================================
# 12. GARANTIR FONTES 5-FAMÍLIAS PARA 2020–2025
# ============================================================

auditoria_fontes = []

fontes = {}


for ano in ANOS:

    fontes[
        ano
    ] = {}


    for indice, (
        grupo,
        nome_base
    ) in enumerate(
        GRUPOS.items(),
        start=1
    ):

        print(
            "\n"
            +
            "#" * 90
        )

        print(
            f"FONTE V3 | {ano} | {grupo} "
            f"({indice}/6)"
        )

        print(
            "#" * 90
        )


        # ----------------------------------------------------
        # 2022 já foi validado na etapa anterior.
        # ----------------------------------------------------

        candidato_2022 = os.path.join(
            PASTA_CANDIDATA_2022,
            f"RAIS_PROFESSORES_2022_{grupo}.parquet"
        )


        if (
            ano
            ==
            2022

            and

            REUTILIZAR_2022_VALIDADO

            and

            fonte_v3_valida(
                candidato_2022
            )
        ):

            caminho_fonte = (
                candidato_2022
            )


            print(
                "Reutilizando 2022 já validado:"
            )

            print(
                caminho_fonte
            )


            pf = pq.ParquetFile(
                caminho_fonte
            )


            auditoria_fontes.append(
                {
                    "Ano":
                        ano,

                    "Grupo":
                        grupo,

                    "Linhas_brutas":
                        np.nan,

                    "Professores_5_familias":
                        int(
                            pf.metadata.num_rows
                        ),

                    "Qtd_colunas_fonte":
                        len(
                            pf.schema_arrow.names
                        ),

                    "Familias_encontradas":
                        "2312, 2313, 2321, 3312, 3321",

                    "UFs_encontradas":
                        "",

                    "Arquivo_fonte":
                        caminho_fonte,

                    "Status":
                        "REUTILIZADO_2022_VALIDADO",
                }
            )


            fontes[
                ano
            ][
                grupo
            ] = caminho_fonte


            continue


        pasta_ano = os.path.join(
            PASTA_FONTE_V3,
            str(
                ano
            )
        )


        os.makedirs(
            pasta_ano,
            exist_ok=True
        )


        caminho_fonte = os.path.join(
            pasta_ano,
            f"RAIS_PROFESSORES_V3_FONTE_{ano}_{grupo}.parquet"
        )


        if (
            fonte_v3_valida(
                caminho_fonte
            )

            and

            not SOBRESCREVER_FONTE
        ):

            pf = pq.ParquetFile(
                caminho_fonte
            )


            print(
                "Fonte já existe:"
            )

            print(
                caminho_fonte
            )


            auditoria_fontes.append(
                {
                    "Ano":
                        ano,

                    "Grupo":
                        grupo,

                    "Linhas_brutas":
                        np.nan,

                    "Professores_5_familias":
                        int(
                            pf.metadata.num_rows
                        ),

                    "Qtd_colunas_fonte":
                        len(
                            pf.schema_arrow.names
                        ),

                    "Familias_encontradas":
                        "",

                    "UFs_encontradas":
                        "",

                    "Arquivo_fonte":
                        caminho_fonte,

                    "Status":
                        "REUTILIZADO",
                }
            )


            fontes[
                ano
            ][
                grupo
            ] = caminho_fonte


            continue


        bruto = localizar_bruto_local(
            ano,
            nome_base
        )


        pasta_temp_extraida = None


        if bruto is None:

            compactado = obter_compactado(
                ano,
                grupo,
                nome_base
            )


            bruto = extrair_compactado(
                compactado,
                ano,
                grupo
            )


            pasta_temp_extraida = (
                os.path.dirname(
                    bruto
                )
            )


        print(
            "Arquivo bruto:"
        )

        print(
            bruto
        )


        try:

            aud = processar_fonte_cinco_familias(
                ano,
                grupo,
                bruto,
                caminho_fonte
            )


            aud[
                "Status"
            ] = "CRIADO"


            auditoria_fontes.append(
                aud
            )


            fontes[
                ano
            ][
                grupo
            ] = caminho_fonte


        finally:

            if (
                pasta_temp_extraida
                is not None

                and

                pasta_temp_extraida.startswith(
                    PASTA_TEMP
                )
            ):

                shutil.rmtree(
                    pasta_temp_extraida,
                    ignore_errors=True
                )


            gc.collect()


df_auditoria_fontes = pd.DataFrame(
    auditoria_fontes
)


df_auditoria_fontes.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "01_auditoria_fontes_5_familias.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 13. LOCALIZAR O ARQUIVO V2 CORRESPONDENTE
# ============================================================

def localizar_v2(
    ano,
    grupo
):

    padroes = [
        os.path.join(
            PASTA_V2,
            "**",
            f"RAIS_MODELO_FINAL_V2_{ano}_{grupo}.parquet"
        ),

        os.path.join(
            PASTA_V2,
            "**",
            f"*V2*{ano}*{grupo}*.parquet"
        ),
    ]


    encontrados = []


    for padrao in padroes:

        encontrados.extend(
            glob.glob(
                padrao,
                recursive=True
            )
        )


    encontrados = sorted(
        set(
            encontrados
        )
    )


    if len(
        encontrados
    ) != 1:

        raise RuntimeError(
            f"{ano} | {grupo}: esperava exatamente "
            f"1 arquivo V2. Encontrei:\n"
            f"{encontrados}"
        )


    return encontrados[
        0
    ]


# ============================================================
# 14. RESOLVER COLUNAS DA FONTE ENRIQUECIDA
# ============================================================

def resolver_colunas_fonte(
    caminho
):

    colunas = pq.ParquetFile(
        caminho
    ).schema_arrow.names


    layout = detectar_layout_colunas(
        colunas
    )


    # Preferir os campos já padronizados criados por nós.
    if (
        "CBO_padronizada"
        in colunas
    ):

        layout[
            "cbo"
        ] = "CBO_padronizada"


    if (
        "Municipio_estabelecimento_codigo"
        in colunas
    ):

        layout[
            "municipio_estab"
        ] = "Municipio_estabelecimento_codigo"


    if (
        "Municipio_trabalhador_codigo"
        in colunas
    ):

        layout[
            "municipio_trab"
        ] = "Municipio_trabalhador_codigo"


    layout[
        "familia"
    ] = "Familia_CBO"


    layout[
        "uf"
    ] = "UF"


    return layout


# ============================================================
# 15. CRIAR NOVAS COLUNAS HARMONIZADAS
# ============================================================

def coluna_ou_na(
    df,
    nome
):

    if (
        nome is None

        or

        nome not in df.columns
    ):

        return pd.Series(
            pd.NA,
            index=df.index,
            dtype="string"
        )


    return df[
        nome
    ]


def criar_novas_variaveis(
    fonte,
    layout
):

    saida = pd.DataFrame(
        index=fonte.index
    )


    saida[
        "Municipio_estabelecimento_codigo"
    ] = extrair_digitos(
        coluna_ou_na(
            fonte,
            layout[
                "municipio_estab"
            ]
        ),
        minimo=6,
        maximo=6
    )


    saida[
        "Municipio_trabalhador_codigo"
    ] = extrair_digitos(
        coluna_ou_na(
            fonte,
            layout[
                "municipio_trab"
            ]
        ),
        minimo=6,
        maximo=6
    )


    saida[
        "CBO_6digitos"
    ] = extrair_digitos(
        coluna_ou_na(
            fonte,
            layout[
                "cbo"
            ]
        ),
        minimo=6,
        maximo=6
    )


    saida[
        "CNAE_2_classe_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "cnae_classe"
            ]
        )
    )


    saida[
        "CNAE_2_subclasse_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "cnae_subclasse"
            ]
        )
    )


    saida[
        "IBGE_subsetor_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "ibge_subsetor"
            ]
        )
    )


    saida[
        "Tipo_admissao_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "tipo_admissao"
            ]
        )
    )


    saida[
        "Mes_admissao_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "mes_admissao"
            ]
        )
    )


    saida[
        "Tipo_estabelecimento_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "tipo_estabelecimento"
            ]
        )
    )


    saida[
        "Indicador_trabalho_parcial_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "trab_parcial"
            ]
        )
    )


    saida[
        "Indicador_trabalho_intermitente_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "trab_intermitente"
            ]
        )
    )


    saida[
        "Vinculo_ativo_31_12_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "vinculo_ativo"
            ]
        )
    )


    saida[
        "Mes_desligamento_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "mes_desligamento"
            ]
        )
    )


    saida[
        "Motivo_desligamento_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "motivo_desligamento"
            ]
        )
    )


    saida[
        "Remuneracao_media_nominal"
    ] = para_numero(
        coluna_ou_na(
            fonte,
            layout[
                "rem_media_nominal"
            ]
        )
    ).astype(
        "float32"
    )


    saida[
        "Remuneracao_dezembro_nominal"
    ] = para_numero(
        coluna_ou_na(
            fonte,
            layout[
                "rem_dezembro_nominal"
            ]
        )
    ).astype(
        "float32"
    )


    saida[
        "Indicador_vinculo_abandonado_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "vinculo_abandonado"
            ]
        )
    )


    saida[
        "Categoria_trabalhador_codigo"
    ] = normalizar_codigo(
        coluna_ou_na(
            fonte,
            layout[
                "categoria_trabalhador"
            ]
        )
    )


    # Tipos compactos das categóricas.
    for coluna in NOVAS_COLUNAS:

        if coluna.startswith(
            "Remuneracao_"
        ):

            continue

        saida[
            coluna
        ] = (
            saida[
                coluna
            ]
            .astype(
                "string"
            )
        )


    return saida


# ============================================================
# 16. ITERADOR DE BLOCOS ALINHADOS
# ============================================================

def iterar_blocos_alinhados(
    parquet_a,
    colunas_a,
    parquet_b,
    colunas_b,
    batch_size=BATCH_SIZE_PARQUET
):

    ita = parquet_a.iter_batches(
        batch_size=batch_size,
        columns=colunas_a
    )

    itb = parquet_b.iter_batches(
        batch_size=batch_size,
        columns=colunas_b
    )


    dfa = None
    dfb = None

    posa = 0
    posb = 0

    fim_a = False
    fim_b = False


    while True:

        if (
            dfa is None

            or

            posa
            >=
            len(
                dfa
            )
        ):

            try:

                dfa = next(
                    ita
                ).to_pandas()

                posa = 0

            except StopIteration:

                fim_a = True


        if (
            dfb is None

            or

            posb
            >=
            len(
                dfb
            )
        ):

            try:

                dfb = next(
                    itb
                ).to_pandas()

                posb = 0

            except StopIteration:

                fim_b = True


        if (
            fim_a

            or

            fim_b
        ):

            if not (
                fim_a
                and
                fim_b
            ):

                raise RuntimeError(
                    "Os dois Parquets terminaram em "
                    "posições diferentes."
                )

            break


        n = min(
            len(
                dfa
            )
            -
            posa,

            len(
                dfb
            )
            -
            posb
        )


        parte_a = (
            dfa
            .iloc[
                posa:
                posa
                +
                n
            ]
            .reset_index(
                drop=True
            )
        )


        parte_b = (
            dfb
            .iloc[
                posb:
                posb
                +
                n
            ]
            .reset_index(
                drop=True
            )
        )


        posa += n
        posb += n


        yield (
            parte_a,
            parte_b
        )


# ============================================================
# 17. VALIDAR ALINHAMENTO LINHA A LINHA
# ============================================================

def comparar_strings(
    a,
    b
):

    aa = normalizar_string_comparacao(
        a
    )

    bb = normalizar_string_comparacao(
        b
    )

    return (
        aa
        ==
        bb
    )


def comparar_numeros(
    a,
    b,
    tolerancia=1e-6
):

    aa = para_numero(
        a
    )

    bb = para_numero(
        b
    )


    ambos_na = (
        aa.isna()
        &
        bb.isna()
    )


    ambos_validos = (
        aa.notna()
        &
        bb.notna()
    )


    iguais = pd.Series(
        False,
        index=aa.index
    )


    iguais.loc[
        ambos_na
    ] = True


    iguais.loc[
        ambos_validos
    ] = np.isclose(
        aa.loc[
            ambos_validos
        ],
        bb.loc[
            ambos_validos
        ],
        rtol=0,
        atol=tolerancia,
        equal_nan=True
    )


    return iguais




def comparar_idade(
    a,
    b,
    tolerancia=1e-6
):
    """
    Compara Idade da fonte RAIS com a V2.

    A auditoria global de 2020–2025 confirmou que a V2 preserva
    as idades válidas e transforma em ausente valores menores que 16.

    Para a validação de alinhamento:
        fonte < 16 e V2 = NaN -> equivalente

    Qualquer outra divergência continua sendo erro.

    Essa regra é usada SOMENTE para validar o alinhamento.
    A coluna Idade da V3 continua sendo copiada diretamente da V2.
    """

    aa = para_numero(a)
    bb = para_numero(b)

    equivalencia_idade_invalida_na = (
        aa.notna()
        &
        aa.lt(16)
        &
        bb.isna()
    )

    ambos_na = (
        aa.isna()
        &
        bb.isna()
    )

    ambos_validos = (
        aa.notna()
        &
        bb.notna()
    )

    iguais = pd.Series(
        False,
        index=aa.index
    )

    iguais.loc[
        ambos_na
    ] = True

    iguais.loc[
        equivalencia_idade_invalida_na
    ] = True

    iguais.loc[
        ambos_validos
    ] = np.isclose(
        aa.loc[
            ambos_validos
        ],
        bb.loc[
            ambos_validos
        ],
        rtol=0,
        atol=tolerancia,
        equal_nan=True
    )

    return (
        iguais,
        int(
            equivalencia_idade_invalida_na.sum()
        )
    )


def comparar_horas_contratuais(
    a,
    b,
    tolerancia=1e-6
):
    """
    Compara Qtd_horas_contratuais da fonte RAIS com a V2.

    A auditoria global de 2020–2025 confirmou a regra presente na V2:

        1 a 44 horas -> valor preservado
        0 horas      -> NaN
        > 44 horas   -> NaN

    Em 2023–2025, o código 99 também aparece e está naturalmente
    abrangido pela regra > 44.

    Essa regra é usada SOMENTE para validar o alinhamento.
    A coluna Qtd_horas_contratuais da V3 continua sendo copiada
    diretamente da V2.
    """

    aa = para_numero(a)
    bb = para_numero(b)

    equivalencia_zero_na = (
        aa.eq(0)
        &
        bb.isna()
    )

    equivalencia_acima_44_na = (
        aa.gt(44)
        &
        bb.isna()
    )

    ambos_na = (
        aa.isna()
        &
        bb.isna()
    )

    ambos_validos = (
        aa.notna()
        &
        bb.notna()
    )

    iguais = pd.Series(
        False,
        index=aa.index
    )

    iguais.loc[
        ambos_na
    ] = True

    iguais.loc[
        equivalencia_zero_na
    ] = True

    iguais.loc[
        equivalencia_acima_44_na
    ] = True

    iguais.loc[
        ambos_validos
    ] = np.isclose(
        aa.loc[
            ambos_validos
        ],
        bb.loc[
            ambos_validos
        ],
        rtol=0,
        atol=tolerancia,
        equal_nan=True
    )

    return (
        iguais,
        int(
            equivalencia_zero_na.sum()
        ),
        int(
            equivalencia_acima_44_na.sum()
        )
    )


def comparar_codigos_numericos(
    a,
    b
):
    """
    Compara códigos categóricos cuja diferença pode ser apenas
    formatação com zeros à esquerda.

    Exemplos equivalentes:
        "01" == "1"
        "02" == "2"

    A comparação é apenas para AUDITORIA DE ALINHAMENTO.
    A V3 continua copiando as variáveis já validadas diretamente
    da V2 e não altera seus códigos.
    """

    a_dig = extrair_digitos(
        a,
        minimo=1,
        maximo=8
    )

    b_dig = extrair_digitos(
        b,
        minimo=1,
        maximo=8
    )

    a_num = pd.to_numeric(
        a_dig,
        errors="coerce"
    )

    b_num = pd.to_numeric(
        b_dig,
        errors="coerce"
    )

    ambos_na = (
        a_num.isna()
        &
        b_num.isna()
    )

    ambos_validos = (
        a_num.notna()
        &
        b_num.notna()
    )

    iguais = pd.Series(
        False,
        index=a_num.index
    )

    iguais.loc[
        ambos_na
    ] = True

    iguais.loc[
        ambos_validos
    ] = (
        a_num.loc[
            ambos_validos
        ]
        ==
        b_num.loc[
            ambos_validos
        ]
    )

    # Contar apenas equivalências em que a representação textual
    # era diferente, mas o código numérico é o mesmo.
    a_txt = normalizar_string_comparacao(
        a_dig
    )

    b_txt = normalizar_string_comparacao(
        b_dig
    )

    equivalencias_formatacao = (
        iguais
        &
        (
            a_txt
            !=
            b_txt
        )
        &
        ambos_validos
    )

    return (
        iguais,
        int(
            equivalencias_formatacao.sum()
        )
    )


def montar_colunas_validacao_alinhamento(
    fonte_schema,
    v2_schema,
    layout
):

    comparacoes = []


    # UF
    if (
        "UF"
        in v2_schema
    ):

        comparacoes.append(
            (
                "UF",
                layout[
                    "uf"
                ],
                "UF",
                "string"
            )
        )


    # Família CBO
    if (
        "Familia_CBO"
        in v2_schema
    ):

        comparacoes.append(
            (
                "Familia_CBO",
                layout[
                    "familia"
                ],
                "Familia_CBO",
                "string"
            )
        )


    # Idade
    if (
        "Idade"
        in v2_schema
    ):

        comparacoes.append(
            (
                "Idade",
                layout[
                    "idade"
                ],
                "Idade",
                "numero_idade"
            )
        )


    # Horas
    if (
        "Qtd_horas_contratuais"
        in v2_schema
    ):

        comparacoes.append(
            (
                "Qtd_horas_contratuais",
                layout[
                    "horas"
                ],
                "Qtd_horas_contratuais",
                "numero_horas"
            )
        )


    # Tempo de emprego
    if (
        "Tempo_emprego_meses"
        in v2_schema
    ):

        comparacoes.append(
            (
                "Tempo_emprego_meses",
                layout[
                    "tempo_emprego"
                ],
                "Tempo_emprego_meses",
                "numero"
            )
        )


    # Sexo
    if (
        "Sexo_codigo"
        in v2_schema
    ):

        comparacoes.append(
            (
                "Sexo_codigo",
                layout[
                    "sexo"
                ],
                "Sexo_codigo",
                "codigo"
            )
        )


    return comparacoes


# ============================================================
# 18. CRIAR V3 A PARTIR DA V2 + NOVAS COVARIÁVEIS
# ============================================================

def construir_v3_arquivo(
    ano,
    grupo,
    caminho_fonte,
    caminho_v2,
    caminho_v3
):

    pf_fonte = pq.ParquetFile(
        caminho_fonte
    )


    pf_v2 = pq.ParquetFile(
        caminho_v2
    )


    n_fonte = int(
        pf_fonte.metadata.num_rows
    )


    n_v2 = int(
        pf_v2.metadata.num_rows
    )


    if (
        n_fonte
        !=
        n_v2
    ):

        raise RuntimeError(
            f"{ano} | {grupo}: quantidade de linhas "
            f"da fonte ({n_fonte:,}) é diferente da "
            f"V2 ({n_v2:,})."
        )


    layout = resolver_colunas_fonte(
        caminho_fonte
    )


    schema_fonte = set(
        pf_fonte.schema_arrow.names
    )


    schema_v2 = list(
        pf_v2.schema_arrow.names
    )


    comparacoes = montar_colunas_validacao_alinhamento(
        schema_fonte,
        schema_v2,
        layout
    )


    if len(
        comparacoes
    ) < 4:

        raise RuntimeError(
            f"{ano} | {grupo}: poucas colunas comuns "
            f"para validar alinhamento: {comparacoes}"
        )


    # Colunas realmente necessárias da fonte.
    colunas_fonte = set(
        valor
        for valor in layout.values()
        if isinstance(valor, str)
        and valor in schema_fonte
    )


    # Acrescentar explicitamente as colunas utilizadas na validação.
    for _, col_fonte, _, _ in comparacoes:

        colunas_fonte.add(
            col_fonte
        )


    colunas_fonte = sorted(
        colunas_fonte
    )


    # Lemos TODAS as colunas da V2, pois ela é a espinha dorsal.
    colunas_v2 = list(
        schema_v2
    )


    # Não permitir colisão silenciosa.
    colisoes = (
        set(
            NOVAS_COLUNAS
        )
        &
        set(
            schema_v2
        )
    )


    if colisoes:

        raise RuntimeError(
            f"{ano} | {grupo}: a V2 já contém "
            f"novas colunas da V3: {sorted(colisoes)}"
        )


    tmp = (
        caminho_v3
        +
        ".tmp"
    )


    if os.path.exists(
        tmp
    ):

        os.remove(
            tmp
        )


    writer = None

    linha_global = 0

    falhas = Counter()

    # Diferenças esperadas decorrentes de limpeza já aplicada na V2.
    # Elas NÃO são ignoradas silenciosamente: são contadas e gravadas
    # nos relatórios de auditoria.
    equivalencias_limpeza = Counter()

    cobertura = {
        coluna: {
            "n":
                0,

            "na":
                0,
        }

        for coluna
        in NOVAS_COLUNAS
    }


    # Para validar que o Y foi copiado sem alteração.
    soma_y_v2 = 0

    soma_y_v3 = 0


    try:

        for (
            fonte,
            v2
        ) in iterar_blocos_alinhados(
            pf_fonte,
            colunas_fonte,
            pf_v2,
            colunas_v2
        ):

            n = len(
                v2
            )


            # -----------------------------------------------
            # Validação linha a linha ANTES de acrescentar
            # qualquer informação.
            # -----------------------------------------------

            for (
                nome,
                col_fonte,
                col_v2,
                tipo
            ) in comparacoes:

                if tipo == "numero_idade":

                    (
                        iguais,
                        qtd_equivalencias_idade_invalida_na
                    ) = comparar_idade(
                        fonte[
                            col_fonte
                        ],
                        v2[
                            col_v2
                        ]
                    )

                    equivalencias_limpeza[
                        "Idade_menor_16_fonte_para_NA_V2"
                    ] += qtd_equivalencias_idade_invalida_na


                elif tipo == "numero_horas":

                    (
                        iguais,
                        qtd_equivalencias_zero_na,
                        qtd_equivalencias_acima_44_na
                    ) = comparar_horas_contratuais(
                        fonte[
                            col_fonte
                        ],
                        v2[
                            col_v2
                        ]
                    )

                    equivalencias_limpeza[
                        "Qtd_horas_0_fonte_para_NA_V2"
                    ] += qtd_equivalencias_zero_na

                    equivalencias_limpeza[
                        "Qtd_horas_acima_44_fonte_para_NA_V2"
                    ] += qtd_equivalencias_acima_44_na


                elif tipo == "numero":

                    iguais = comparar_numeros(
                        fonte[
                            col_fonte
                        ],
                        v2[
                            col_v2
                        ]
                    )


                elif tipo == "codigo":

                    (
                        iguais,
                        qtd_equivalencias_codigo
                    ) = comparar_codigos_numericos(
                        fonte[
                            col_fonte
                        ],
                        v2[
                            col_v2
                        ]
                    )

                    equivalencias_limpeza[
                        f"{nome}_formatacao_codigo"
                    ] += qtd_equivalencias_codigo


                else:

                    iguais = comparar_strings(
                        fonte[
                            col_fonte
                        ],
                        v2[
                            col_v2
                        ]
                    )


                qtd_falhas = int(
                    (
                        ~iguais
                    )
                    .sum()
                )


                falhas[
                    nome
                ] += qtd_falhas


                if qtd_falhas > 0:

                    exemplos_idx = (
                        np.flatnonzero(
                            (
                                ~iguais
                            )
                            .to_numpy()
                        )[
                            :5
                        ]
                    )


                    exemplos = []


                    for idx in exemplos_idx:

                        exemplos.append(
                            {
                                "linha":
                                    int(
                                        linha_global
                                        +
                                        idx
                                    ),

                                "fonte":
                                    str(
                                        fonte.loc[
                                            idx,
                                            col_fonte
                                        ]
                                    ),

                                "v2":
                                    str(
                                        v2.loc[
                                            idx,
                                            col_v2
                                        ]
                                    ),
                            }
                        )


                    raise RuntimeError(
                        f"{ano} | {grupo}: alinhamento "
                        f"falhou em {nome}. "
                        f"Falhas neste bloco={qtd_falhas}. "
                        f"Exemplos={exemplos}"
                    )


            # -----------------------------------------------
            # Criar covariáveis novas
            # -----------------------------------------------

            novas = criar_novas_variaveis(
                fonte,
                layout
            )


            for coluna in NOVAS_COLUNAS:

                cobertura[
                    coluna
                ][
                    "n"
                ] += n


                cobertura[
                    coluna
                ][
                    "na"
                ] += int(
                    novas[
                        coluna
                    ]
                    .isna()
                    .sum()
                )


            saida = (
                v2
                .copy()
            )


            for coluna in NOVAS_COLUNAS:

                saida[
                    coluna
                ] = novas[
                    coluna
                ].values


            # Segurança adicional:
            # Y_doenca deve ter vindo exclusivamente da V2.
            if (
                "Y_doenca"
                in v2.columns
            ):

                y_v2 = pd.to_numeric(
                    v2[
                        "Y_doenca"
                    ],
                    errors="raise"
                )


                y_v3 = pd.to_numeric(
                    saida[
                        "Y_doenca"
                    ],
                    errors="raise"
                )


                if not np.array_equal(
                    y_v2.to_numpy(),
                    y_v3.to_numpy()
                ):

                    raise RuntimeError(
                        f"{ano} | {grupo}: Y_doenca foi "
                        "alterado durante a criação da V3."
                    )


                soma_y_v2 += int(
                    y_v2.sum()
                )


                soma_y_v3 += int(
                    y_v3.sum()
                )


            tabela = pa.Table.from_pandas(
                saida,
                preserve_index=False
            )


            if writer is None:

                writer = pq.ParquetWriter(
                    tmp,
                    tabela.schema,
                    compression="snappy"
                )


            writer.write_table(
                tabela
            )


            linha_global += n


            print(
                f"{ano} | {grupo} | "
                f"{linha_global:,}/{n_v2:,}"
            )


            del (
                fonte,
                v2,
                novas,
                saida,
                tabela
            )


            gc.collect()


    finally:

        if writer is not None:

            writer.close()


    if (
        linha_global
        !=
        n_v2
    ):

        raise RuntimeError(
            f"{ano} | {grupo}: foram processadas "
            f"{linha_global:,} linhas, mas a V2 possui "
            f"{n_v2:,}."
        )


    pf_tmp = pq.ParquetFile(
        tmp
    )


    if int(
        pf_tmp.metadata.num_rows
    ) != n_v2:

        raise RuntimeError(
            f"{ano} | {grupo}: V3 temporária possui "
            f"{pf_tmp.metadata.num_rows:,} linhas; "
            f"esperado {n_v2:,}."
        )


    faltantes = (
        set(
            NOVAS_COLUNAS
        )
        -
        set(
            pf_tmp.schema_arrow.names
        )
    )


    if faltantes:

        raise RuntimeError(
            f"{ano} | {grupo}: novas colunas ausentes: "
            f"{sorted(faltantes)}"
        )


    if (
        soma_y_v2
        !=
        soma_y_v3
    ):

        raise RuntimeError(
            f"{ano} | {grupo}: soma de Y_doenca mudou: "
            f"{soma_y_v2} vs {soma_y_v3}."
        )


    if os.path.exists(
        caminho_v3
    ):

        os.remove(
            caminho_v3
        )


    os.replace(
        tmp,
        caminho_v3
    )


    linhas_cobertura = []


    for coluna in NOVAS_COLUNAS:

        total = cobertura[
            coluna
        ][
            "n"
        ]


        nulos = cobertura[
            coluna
        ][
            "na"
        ]


        linhas_cobertura.append(
            {
                "Ano":
                    ano,

                "Grupo":
                    grupo,

                "Variavel":
                    coluna,

                "N":
                    total,

                "N_nulos":
                    nulos,

                "Percentual_nulo":
                    (
                        nulos
                        /
                        total
                        *
                        100
                    )
                    if total
                    else np.nan,
            }
        )


    auditoria = {
        "Ano":
            ano,

        "Grupo":
            grupo,

        "N_fonte":
            n_fonte,

        "N_V2":
            n_v2,

        "N_V3":
            int(
                pq.ParquetFile(
                    caminho_v3
                )
                .metadata.num_rows
            ),

        "Y_positivos_V2":
            soma_y_v2,

        "Y_positivos_V3":
            soma_y_v3,

        "Falhas_alinhamento":
            int(
                sum(
                    falhas.values()
                )
            ),

        "Equivalencias_Idade_menor_16_fonte_para_NA_V2":
            int(
                equivalencias_limpeza.get(
                    "Idade_menor_16_fonte_para_NA_V2",
                    0
                )
            ),

        "Equivalencias_horas_0_fonte_para_NA_V2":
            int(
                equivalencias_limpeza.get(
                    "Qtd_horas_0_fonte_para_NA_V2",
                    0
                )
            ),

        "Equivalencias_horas_acima_44_fonte_para_NA_V2":
            int(
                equivalencias_limpeza.get(
                    "Qtd_horas_acima_44_fonte_para_NA_V2",
                    0
                )
            ),

        "Equivalencias_Sexo_codigo_formatacao":
            int(
                equivalencias_limpeza.get(
                    "Sexo_codigo_formatacao_codigo",
                    0
                )
            ),

        "Colunas_validacao_alinhamento":
            ", ".join(
                nome
                for (
                    nome,
                    _,
                    _,
                    _
                )
                in comparacoes
            ),

        "Arquivo_fonte":
            caminho_fonte,

        "Arquivo_V2":
            caminho_v2,

        "Arquivo_V3":
            caminho_v3,
    }


    return (
        auditoria,
        linhas_cobertura
    )


# ============================================================
# 19. CRIAR OS 36 PARQUETS DA V3
# ============================================================

auditoria_alinhamento = []

auditoria_cobertura = []


for ano in ANOS:

    pasta_ano_v3 = os.path.join(
        PASTA_V3,
        str(
            ano
        )
    )


    os.makedirs(
        pasta_ano_v3,
        exist_ok=True
    )


    for indice, grupo in enumerate(
        GRUPOS.keys(),
        start=1
    ):

        print(
            "\n"
            +
            "=" * 100
        )

        print(
            f"V3 | {ano} | {grupo} ({indice}/6)"
        )

        print(
            "=" * 100
        )


        caminho_fonte = (
            fontes[
                ano
            ][
                grupo
            ]
        )


        caminho_v2 = localizar_v2(
            ano,
            grupo
        )


        caminho_v3 = os.path.join(
            pasta_ano_v3,
            f"RAIS_MODELO_FINAL_V3_{ano}_{grupo}.parquet"
        )


        if (
            os.path.exists(
                caminho_v3
            )

            and

            not SOBRESCREVER_V3
        ):

            pf_v3 = pq.ParquetFile(
                caminho_v3
            )

            pf_v2 = pq.ParquetFile(
                caminho_v2
            )


            schema_ok = (
                set(
                    NOVAS_COLUNAS
                )
                .issubset(
                    set(
                        pf_v3.schema_arrow.names
                    )
                )
            )


            linhas_ok = (
                int(
                    pf_v3.metadata.num_rows
                )
                ==
                int(
                    pf_v2.metadata.num_rows
                )
            )


            if (
                schema_ok
                and
                linhas_ok
            ):

                print(
                    "V3 já existe e passou na verificação "
                    "básica. Reutilizando."
                )


                auditoria_alinhamento.append(
                    {
                        "Ano":
                            ano,

                        "Grupo":
                            grupo,

                        "N_fonte":
                            int(
                                pq.ParquetFile(
                                    caminho_fonte
                                )
                                .metadata.num_rows
                            ),

                        "N_V2":
                            int(
                                pf_v2.metadata.num_rows
                            ),

                        "N_V3":
                            int(
                                pf_v3.metadata.num_rows
                            ),

                        "Y_positivos_V2":
                            np.nan,

                        "Y_positivos_V3":
                            np.nan,

                        "Falhas_alinhamento":
                            0,

                        "Equivalencias_horas_0_fonte_para_NA_V2":
                            np.nan,

                        "Colunas_validacao_alinhamento":
                            "REUTILIZADO",

                        "Arquivo_fonte":
                            caminho_fonte,

                        "Arquivo_V2":
                            caminho_v2,

                        "Arquivo_V3":
                            caminho_v3,

                        "Status":
                            "REUTILIZADO",
                    }
                )


                continue


        aud, cobertura = construir_v3_arquivo(
            ano,
            grupo,
            caminho_fonte,
            caminho_v2,
            caminho_v3
        )


        aud[
            "Status"
        ] = "CRIADO_E_VALIDADO"


        auditoria_alinhamento.append(
            aud
        )


        auditoria_cobertura.extend(
            cobertura
        )


        pd.DataFrame(
            auditoria_alinhamento
        ).to_csv(
            os.path.join(
                PASTA_RESULTADOS,
                "02_auditoria_alinhamento_V2_V3_parcial.csv"
            ),
            index=False,
            encoding="utf-8-sig"
        )


# ============================================================
# 20. AUDITORIA FINAL V2 x V3
# ============================================================

df_alinhamento = pd.DataFrame(
    auditoria_alinhamento
)


df_cobertura = pd.DataFrame(
    auditoria_cobertura
)


df_alinhamento.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "02_auditoria_alinhamento_V2_V3.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


if len(
    df_cobertura
):

    df_cobertura.to_csv(
        os.path.join(
            PASTA_RESULTADOS,
            "03_cobertura_novas_variaveis_por_arquivo.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 21. COMPARAR V2 E V3 POR ANO
# ============================================================

def resumir_arquivo_modelo(
    caminho,
    ano,
    grupo
):

    pf = pq.ParquetFile(
        caminho
    )


    colunas_disponiveis = set(
        pf.schema_arrow.names
    )


    essenciais = [
        "UF",
        "Familia_CBO",
        "Y_doenca",
    ]


    faltantes = (
        set(
            essenciais
        )
        -
        colunas_disponiveis
    )


    if faltantes:

        raise RuntimeError(
            f"Faltam {faltantes} em {caminho}"
        )


    total = 0

    y_pos = 0

    cont_uf = Counter()

    cont_fam = Counter()

    cont_uf_fam = Counter()


    for batch in pf.iter_batches(
        batch_size=BATCH_SIZE_PARQUET,
        columns=essenciais
    ):

        df = batch.to_pandas()


        total += len(
            df
        )


        y = pd.to_numeric(
            df[
                "Y_doenca"
            ],
            errors="raise"
        )


        y_pos += int(
            (
                y
                ==
                1
            )
            .sum()
        )


        uf = (
            df[
                "UF"
            ]
            .astype(
                "string"
            )
            .str.strip()
            .str.upper()
        )


        fam = (
            df[
                "Familia_CBO"
            ]
            .astype(
                "string"
            )
            .str.extract(
                r"(\d{4})",
                expand=False
            )
        )


        for valor, qtd in (
            uf
            .value_counts(
                dropna=False
            )
            .items()
        ):

            cont_uf[
                str(
                    valor
                )
            ] += int(
                qtd
            )


        for valor, qtd in (
            fam
            .value_counts(
                dropna=False
            )
            .items()
        ):

            cont_fam[
                str(
                    valor
                )
            ] += int(
                qtd
            )


        temp = pd.DataFrame(
            {
                "UF":
                    uf,

                "Familia_CBO":
                    fam,
            }
        )


        for (
            chave,
            qtd
        ) in (
            temp
            .groupby(
                [
                    "UF",
                    "Familia_CBO"
                ],
                dropna=False
            )
            .size()
            .items()
        ):

            cont_uf_fam[
                (
                    str(
                        chave[
                            0
                        ]
                    ),
                    str(
                        chave[
                            1
                        ]
                    )
                )
            ] += int(
                qtd
            )


        del (
            df,
            temp
        )


    return {
        "Ano":
            ano,

        "Grupo":
            grupo,

        "N":
            total,

        "Y_positivos":
            y_pos,

        "Prevalencia_pct":
            (
                y_pos
                /
                total
                *
                100
            )
            if total
            else np.nan,

        "cont_uf":
            cont_uf,

        "cont_fam":
            cont_fam,

        "cont_uf_fam":
            cont_uf_fam,
    }


validacao_arquivos = []

resumos_v2 = []

resumos_v3 = []

cont_v2_uf = Counter()

cont_v3_uf = Counter()

cont_v2_fam = Counter()

cont_v3_fam = Counter()

cont_v2_uf_fam = Counter()

cont_v3_uf_fam = Counter()


for ano in ANOS:

    for grupo in GRUPOS:

        v2 = localizar_v2(
            ano,
            grupo
        )


        v3 = os.path.join(
            PASTA_V3,
            str(
                ano
            ),
            f"RAIS_MODELO_FINAL_V3_{ano}_{grupo}.parquet"
        )


        if not os.path.exists(
            v3
        ):

            raise FileNotFoundError(
                v3
            )


        rv2 = resumir_arquivo_modelo(
            v2,
            ano,
            grupo
        )


        rv3 = resumir_arquivo_modelo(
            v3,
            ano,
            grupo
        )


        resumos_v2.append(
            {
                chave:
                    valor

                for chave, valor
                in rv2.items()

                if not chave.startswith(
                    "cont_"
                )
            }
        )


        resumos_v3.append(
            {
                chave:
                    valor

                for chave, valor
                in rv3.items()

                if not chave.startswith(
                    "cont_"
                )
            }
        )


        for k, v in rv2[
            "cont_uf"
        ].items():

            cont_v2_uf[
                (
                    ano,
                    k
                )
            ] += v


        for k, v in rv3[
            "cont_uf"
        ].items():

            cont_v3_uf[
                (
                    ano,
                    k
                )
            ] += v


        for k, v in rv2[
            "cont_fam"
        ].items():

            cont_v2_fam[
                (
                    ano,
                    k
                )
            ] += v


        for k, v in rv3[
            "cont_fam"
        ].items():

            cont_v3_fam[
                (
                    ano,
                    k
                )
            ] += v


        for k, v in rv2[
            "cont_uf_fam"
        ].items():

            cont_v2_uf_fam[
                (
                    ano,
                    k[
                        0
                    ],
                    k[
                        1
                    ]
                )
            ] += v


        for k, v in rv3[
            "cont_uf_fam"
        ].items():

            cont_v3_uf_fam[
                (
                    ano,
                    k[
                        0
                    ],
                    k[
                        1
                    ]
                )
            ] += v


        validacao_arquivos.append(
            {
                "Ano":
                    ano,

                "Grupo":
                    grupo,

                "N_V2":
                    rv2[
                        "N"
                    ],

                "N_V3":
                    rv3[
                        "N"
                    ],

                "Diferenca_N":
                    rv3[
                        "N"
                    ]
                    -
                    rv2[
                        "N"
                    ],

                "Y_positivos_V2":
                    rv2[
                        "Y_positivos"
                    ],

                "Y_positivos_V3":
                    rv3[
                        "Y_positivos"
                    ],

                "Diferenca_Y":
                    rv3[
                        "Y_positivos"
                    ]
                    -
                    rv2[
                        "Y_positivos"
                    ],
            }
        )


df_validacao_arquivos = pd.DataFrame(
    validacao_arquivos
)


df_resumo_v2 = pd.DataFrame(
    resumos_v2
)


df_resumo_v3 = pd.DataFrame(
    resumos_v3
)


# ============================================================
# 22. RESUMO POR ANO
# ============================================================

resumo_ano_v2 = (
    df_resumo_v2
    .groupby(
        "Ano",
        as_index=False
    )
    .agg(
        N_V2=(
            "N",
            "sum"
        ),

        Y_positivos_V2=(
            "Y_positivos",
            "sum"
        ),
    )
)


resumo_ano_v3 = (
    df_resumo_v3
    .groupby(
        "Ano",
        as_index=False
    )
    .agg(
        N_V3=(
            "N",
            "sum"
        ),

        Y_positivos_V3=(
            "Y_positivos",
            "sum"
        ),
    )
)


df_resumo_ano = resumo_ano_v2.merge(
    resumo_ano_v3,
    on="Ano",
    how="outer"
)


df_resumo_ano[
    "Diferenca_N"
] = (
    df_resumo_ano[
        "N_V3"
    ]
    -
    df_resumo_ano[
        "N_V2"
    ]
)


df_resumo_ano[
    "Diferenca_Y"
] = (
    df_resumo_ano[
        "Y_positivos_V3"
    ]
    -
    df_resumo_ano[
        "Y_positivos_V2"
    ]
)


df_resumo_ano[
    "Prevalencia_V2_pct"
] = (
    df_resumo_ano[
        "Y_positivos_V2"
    ]
    /
    df_resumo_ano[
        "N_V2"
    ]
    *
    100
)


df_resumo_ano[
    "Prevalencia_V3_pct"
] = (
    df_resumo_ano[
        "Y_positivos_V3"
    ]
    /
    df_resumo_ano[
        "N_V3"
    ]
    *
    100
)


# ============================================================
# 23. COMPARAÇÕES AGREGADAS DE UF/FAMÍLIA
# ============================================================

def counter_para_df(
    counter_a,
    counter_b,
    nomes_chaves,
    nome_a="N_V2",
    nome_b="N_V3"
):

    chaves = sorted(
        set(
            counter_a.keys()
        )
        |
        set(
            counter_b.keys()
        )
    )


    linhas = []


    for chave in chaves:

        if not isinstance(
            chave,
            tuple
        ):

            chave = (
                chave,
            )


        linha = {
            nome:
                valor

            for nome, valor
            in zip(
                nomes_chaves,
                chave
            )
        }


        linha[
            nome_a
        ] = int(
            counter_a.get(
                chave,
                0
            )
        )


        linha[
            nome_b
        ] = int(
            counter_b.get(
                chave,
                0
            )
        )


        linha[
            "Diferenca"
        ] = (
            linha[
                nome_b
            ]
            -
            linha[
                nome_a
            ]
        )


        linhas.append(
            linha
        )


    return pd.DataFrame(
        linhas
    )


df_cmp_uf = counter_para_df(
    cont_v2_uf,
    cont_v3_uf,
    [
        "Ano",
        "UF"
    ]
)


df_cmp_fam = counter_para_df(
    cont_v2_fam,
    cont_v3_fam,
    [
        "Ano",
        "Familia_CBO"
    ]
)


df_cmp_uf_fam = counter_para_df(
    cont_v2_uf_fam,
    cont_v3_uf_fam,
    [
        "Ano",
        "UF",
        "Familia_CBO"
    ]
)


# ============================================================
# 24. COBERTURA FINAL DAS NOVAS VARIÁVEIS
# ============================================================

cobertura_final = []


for ano in ANOS:

    for grupo in GRUPOS:

        caminho = os.path.join(
            PASTA_V3,
            str(
                ano
            ),
            f"RAIS_MODELO_FINAL_V3_{ano}_{grupo}.parquet"
        )


        pf = pq.ParquetFile(
            caminho
        )


        for batch in pf.iter_batches(
            batch_size=BATCH_SIZE_PARQUET,
            columns=NOVAS_COLUNAS
        ):

            df = batch.to_pandas()


            for coluna in NOVAS_COLUNAS:

                cobertura_final.append(
                    {
                        "Ano":
                            ano,

                        "Grupo":
                            grupo,

                        "Variavel":
                            coluna,

                        "N":
                            len(
                                df
                            ),

                        "N_nulos":
                            int(
                                df[
                                    coluna
                                ]
                                .isna()
                                .sum()
                            ),
                    }
                )


            del df


df_cobertura_final = pd.DataFrame(
    cobertura_final
)


df_cobertura_final = (
    df_cobertura_final
    .groupby(
        [
            "Ano",
            "Variavel"
        ],
        as_index=False
    )
    .agg(
        N=(
            "N",
            "sum"
        ),

        N_nulos=(
            "N_nulos",
            "sum"
        ),
    )
)


df_cobertura_final[
    "Percentual_nulo"
] = (
    df_cobertura_final[
        "N_nulos"
    ]
    /
    df_cobertura_final[
        "N"
    ]
    *
    100
)


# ============================================================
# 25. SCHEMA FINAL
# ============================================================

schemas_v3 = []


for ano in ANOS:

    for grupo in GRUPOS:

        caminho = os.path.join(
            PASTA_V3,
            str(
                ano
            ),
            f"RAIS_MODELO_FINAL_V3_{ano}_{grupo}.parquet"
        )


        pf = pq.ParquetFile(
            caminho
        )


        assinatura = "|".join(
            f"{campo.name}:{campo.type}"
            for campo
            in pf.schema_arrow
        )


        schema_id = hashlib.sha256(
            assinatura.encode(
                "utf-8"
            )
        ).hexdigest()[
            :12
        ]


        schemas_v3.append(
            {
                "Ano":
                    ano,

                "Grupo":
                    grupo,

                "N":
                    int(
                        pf.metadata.num_rows
                    ),

                "Qtd_colunas":
                    len(
                        pf.schema_arrow.names
                    ),

                "Schema_ID":
                    schema_id,

                "Colunas":
                    " | ".join(
                        pf.schema_arrow.names
                    ),
            }
        )


df_schema_v3 = pd.DataFrame(
    schemas_v3
)


# ============================================================
# 26. VALIDAÇÃO FINAL
# ============================================================

total_v2 = int(
    df_resumo_ano[
        "N_V2"
    ]
    .sum()
)


total_v3 = int(
    df_resumo_ano[
        "N_V3"
    ]
    .sum()
)


y_v2 = int(
    df_resumo_ano[
        "Y_positivos_V2"
    ]
    .sum()
)


y_v3 = int(
    df_resumo_ano[
        "Y_positivos_V3"
    ]
    .sum()
)


condicoes = {

    "36_arquivos_V3":
        len(
            df_validacao_arquivos
        )
        ==
        36,

    "totais_iguais":
        total_v2
        ==
        total_v3,

    "Y_iguais":
        y_v2
        ==
        y_v3,

    "arquivo_a_arquivo_sem_diferenca":
        (
            df_validacao_arquivos[
                [
                    "Diferenca_N",
                    "Diferenca_Y"
                ]
            ]
            ==
            0
        )
        .all()
        .all(),

    "UF_sem_diferenca":
        (
            df_cmp_uf[
                "Diferenca"
            ]
            ==
            0
        )
        .all(),

    "familia_sem_diferenca":
        (
            df_cmp_fam[
                "Diferenca"
            ]
            ==
            0
        )
        .all(),

    "UF_familia_sem_diferenca":
        (
            df_cmp_uf_fam[
                "Diferenca"
            ]
            ==
            0
        )
        .all(),

    "municipio_estabelecimento_completo":
        (
            df_cobertura_final.loc[
                df_cobertura_final[
                    "Variavel"
                ]
                ==
                "Municipio_estabelecimento_codigo",
                "N_nulos"
            ]
            ==
            0
        )
        .all(),
}


validacao_final = pd.DataFrame(
    [
        {
            "Teste":
                chave,

            "Aprovado":
                bool(
                    valor
                )
        }

        for chave, valor
        in condicoes.items()
    ]
)


APROVADA = all(
    condicoes.values()
)


# ============================================================
# 27. SALVAR RELATÓRIOS
# ============================================================

df_validacao_arquivos.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "04_validacao_V2_V3_por_arquivo.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


df_resumo_ano.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "05_validacao_V2_V3_por_ano.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


df_cmp_uf.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "06_validacao_V2_V3_por_UF.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


df_cmp_fam.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "07_validacao_V2_V3_por_familia_CBO.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


df_cmp_uf_fam.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "08_validacao_V2_V3_UF_familia_CBO.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


df_cobertura_final.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "09_cobertura_novas_variaveis_V3.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


df_schema_v3.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "10_schema_V3.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


validacao_final.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "11_VALIDACAO_FINAL_V3.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


manifesto = {
    "base":
        "RAIS_BASE_MODELO_FINAL_V3",

    "anos":
        ANOS,

    "familias_CBO":
        list(
            FAMILIAS_CBO.keys()
        ),

    "total_V2":
        total_v2,

    "total_V3":
        total_v3,

    "Y_positivos_V2":
        y_v2,

    "Y_positivos_V3":
        y_v3,

    "novas_colunas":
        NOVAS_COLUNAS,

    "Y_doenca":
        "copiado diretamente da V2; não reconstruído",

    "campos_afastamento_como_preditores":
        False,

    "validacao_aprovada":
        APROVADA,

    "testes":
        condicoes,
}


with open(
    os.path.join(
        PASTA_RESULTADOS,
        "12_manifesto_V3.json"
    ),
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        manifesto,
        arquivo,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 28. EXIBIR RESULTADOS
# ============================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "VALIDAÇÃO FINAL DA V3"
)

print(
    "=" * 100
)


display(
    validacao_final
)


print(
    "\nRESUMO POR ANO"
)

display(
    df_resumo_ano
)


print(
    "\nCOBERTURA DAS NOVAS VARIÁVEIS"
)

display(
    df_cobertura_final
)


print(
    "\nTOTAL V2:"
)

print(
    f"{total_v2:,}"
)


print(
    "\nTOTAL V3:"
)

print(
    f"{total_v3:,}"
)


print(
    "\nY_doenca positivos V2:"
)

print(
    f"{y_v2:,}"
)


print(
    "\nY_doenca positivos V3:"
)

print(
    f"{y_v3:,}"
)


if APROVADA:

    print(
        "\n"
        +
        "#" * 100
    )

    print(
        "V3 APROVADA."
    )

    print(
        "A população, o Y_doenca e as distribuições "
        "de UF/Família CBO são idênticos à V2."
    )

    print(
        "Podemos avançar para os experimentos "
        "M0 → M5."
    )

    print(
        "#" * 100
    )

else:

    print(
        "\n"
        +
        "#" * 100
    )

    print(
        "V3 NÃO APROVADA."
    )

    print(
        "Não treine novos modelos ainda."
    )

    print(
        "Revise 11_VALIDACAO_FINAL_V3.csv "
        "e os relatórios 04–09."
    )

    print(
        "#" * 100
    )


print(
    "\nBase V3:"
)

print(
    PASTA_V3
)


print(
    "\nRelatórios:"
)

print(
    PASTA_RESULTADOS
)


In [ ]:
import json
import os
import numpy as np


def tornar_json_serializavel(obj):

    if isinstance(obj, np.bool_):
        return bool(obj)

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, dict):
        return {
            str(chave): tornar_json_serializavel(valor)
            for chave, valor in obj.items()
        }

    if isinstance(obj, (list, tuple)):
        return [
            tornar_json_serializavel(valor)
            for valor in obj
        ]

    return obj


manifesto_seguro = tornar_json_serializavel(
    manifesto
)


with open(
    os.path.join(
        PASTA_RESULTADOS,
        "12_manifesto_V3.json"
    ),
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        manifesto_seguro,
        arquivo,
        ensure_ascii=False,
        indent=2
    )


print("Manifesto salvo com sucesso.")


print("\nVALIDAÇÃO FINAL DA V3")
display(validacao_final)


print("\nRESUMO POR ANO")
display(df_resumo_ano)


print("\nTOTAL V2:", f"{total_v2:,}")
print("TOTAL V3:", f"{total_v3:,}")
print("Y positivos V2:", f"{y_v2:,}")
print("Y positivos V3:", f"{y_v3:,}")


if bool(APROVADA):

    print("\nV3 APROVADA.")

else:

    print("\nV3 NÃO APROVADA.")